# Étape 4 — Optimisation et évaluation


## 1. Chargement des données et modèles de référence

On recharge les données préparées à l'étape 2 (`train.csv` / `test.csv` + `preprocessor.pkl`), ainsi que les modèles de référence (mêmes hyperparamètres qu'à l'étape 3, `random_state` identique → résultats reproductibles) pour construire un tableau **avant / après optimisation**.

Le baseline de la Logistic Regression est celui sauvegardé à l'étape 3 ; les deux autres sont ré-instanciés puis ré-entraînés à l'identique.


In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

sns.set_theme(style="whitegrid")

ROOT_DIR = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
import sys; sys.path.insert(0, str(ROOT_DIR / "src"))
DATA_PROCESSED_DIR = ROOT_DIR / "data" / "processed"
MODELS_DIR = ROOT_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)

train_df = pd.read_csv(DATA_PROCESSED_DIR / "train.csv")
test_df = pd.read_csv(DATA_PROCESSED_DIR / "test.csv")
TARGET = "Mental_Health_Risk"

y_train = train_df[TARGET]
y_test = test_df[TARGET]

preprocessor = joblib.load(DATA_PROCESSED_DIR / "preprocessor.pkl")
X_train = preprocessor.transform(train_df.drop(columns=[TARGET]))
X_test = preprocessor.transform(test_df.drop(columns=[TARGET]))
train_df = pd.read_csv(DATA_PROCESSED_DIR / "train.csv")
test_df = pd.read_csv(DATA_PROCESSED_DIR / "test.csv")
TARGET = "Mental_Health_Risk"

y_train = train_df[TARGET]
y_test = test_df[TARGET]

preprocessor = joblib.load(DATA_PROCESSED_DIR / "preprocessor.pkl")
X_train = preprocessor.transform(train_df.drop(columns=[TARGET]))
X_test = preprocessor.transform(test_df.drop(columns=[TARGET]))

RISK_ORDER = ["Low", "Medium", "High"]

BASELINES = {
    "Logistic Regression": joblib.load(MODELS_DIR / "logistic_regression_baseline.pkl"),
    "SVM": Pipeline([("classifier", SVC(random_state=42))]),
    "Random Forest": Pipeline([("classifier", RandomForestClassifier(n_estimators=100, random_state=42))]),
}

for model in BASELINES.values():
    if not hasattr(model, "classes_"):
        model.fit(X_train, y_train)


def evaluate(model, X, y_true, name):

    pred = model.predict(X)
   
    if hasattr(model, "predict_proba"):
        auc = roc_auc_score(y_true, model.predict_proba(X),
                            multi_class="ovr", average="weighted")
        auc = np.nan
    return {
        "Model": name,
        "Recall": recall_score(y_true, pred, average="weighted"),
        "Precision": precision_score(y_true, pred, average="weighted"),
        "F1 Score": f1_score(y_true, pred, average="weighted"),
        "Accuracy": accuracy_score(y_true, pred),
        "ROC AUC": auc,
    }


baseline_results = pd.DataFrame(
    [evaluate(model, X_test, y_test, name) for name, model in BASELINES.items()]
)
print(baseline_results.round(4).to_string(index=False))

Train : (1600, 21) | Test : (400, 21)


              Model  Recall  Precision  F1 Score  Accuracy  ROC AUC
Logistic Regression  0.7575     0.7576    0.7575    0.7575   0.9150
                SVM  0.7225     0.7205    0.7214    0.7225      NaN
      Random Forest  0.6775     0.6696    0.6721    0.6775   0.8528


## 2. Optimisation de la Logistic Regression — GridSearchCV


In [2]:
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
}

grid_search = GridSearchCV(
    BASELINES["Logistic Regression"],
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=5,
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print(f"Meilleurs paramètres : {grid_search.best_params_}")
print(f"F1 pondéré (validation croisée, train) : {grid_search.best_score_:.4f}")

grid_results = (
    pd.DataFrame(grid_search.cv_results_)
    [["param_classifier__C", "mean_test_score", "rank_test_score"]]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)
grid_results.head(10)

Meilleurs paramètres : {'classifier__C': 1}
F1 pondéré (validation croisée, train) : 0.7533


,param_classifier__C,mean_test_score,rank_test_score
0,1.00,0.753270,1
1,10.00,0.752085,2
2,100.00,0.752085,2
3,0.10,0.741476,4
4,0.01,0.701498,5


## 3. Optimisation du SVM — RandomizedSearchCV


In [3]:
param_distributions = {
    "classifier__C": [0.1, 1, 10, 100],
    "classifier__gamma": ["scale", 0.01, 0.1, 1],
    "classifier__kernel": ["rbf", "linear"],
}

random_search = RandomizedSearchCV(
    BASELINES["SVM"],
    param_distributions=param_distributions,
    n_iter=12,
    scoring="f1_weighted",
    cv=5,
    random_state=42,
    n_jobs=-1,
)
random_search.fit(X_train, y_train)

print(f"Meilleurs paramètres : {random_search.best_params_}")
print(f"F1 pondéré (validation croisée, train) : {random_search.best_score_:.4f}")

Meilleurs paramètres : {'classifier__kernel': 'linear', 'classifier__gamma': 1, 'classifier__C': 1}
F1 pondéré (validation croisée, train) : 0.7498


## 4. Évaluation des modèles optimisés sur le jeu de test indépendant
